# Notebook 04 — Validación cruzada espacial vs aleatoria

Comparamos dos esquemas de CV: aleatoria $k=4$ y espacial por bloques 2×2. La hipótesis es que la CV aleatoria *subestima* el RMSE al contaminar cada fold con vecinos espaciales del entrenamiento, mientras que la espacial entrega una estimación más honesta del desempeño en zonas no vistas. Esto se alinea con la *Area of Applicability* de Meyer & Pebesma (2021).

## Compatibilidad con Google Colab

La siguiente celda detecta si el notebook se ejecuta en Colab y, en ese caso, instala `gpboost` y `shap` y clona el repositorio para tener acceso a los datos derivados (`data/`) y a los modelos entrenados (`modelos/`). En Jupyter local no hace nada.

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Entorno Colab — instalando dependencias...')
    !pip -q install gpboost shap
    if not os.path.isdir('/content/repo'):
        !git clone -q https://github.com/linaq11/examen-2-gpboost.git /content/repo
    os.chdir('/content/repo')
    print('CWD:', os.getcwd())
else:
    # En local: asume que el notebook está en repo/notebooks/
    here = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
    if os.path.basename(here) == 'notebooks':
        os.chdir(os.path.dirname(here))
    print('CWD:', os.getcwd())

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gpboost as gpb

np.random.seed(1)
DATA = 'data'
MOD  = 'modelos'
FIG  = 'figuras'
os.makedirs(FIG, exist_ok=True)
from sklearn.model_selection import KFold

## 1. Datos y tabla de resultados

In [ ]:
df = pd.read_csv(os.path.join(DATA, 'simulado_train_seed1.csv'))
X      = df[['X1','X2']].values
y      = df['y'].values
coords = df[['s_1','s_2']].values

tabla = pd.read_csv(os.path.join(DATA, 'tabla_cv_espacial.csv'))
tabla

## 2. Visualización del gap CV/test

El RMSE del test grid 50×50 es 0.6073 (referencia). La CV aleatoria queda por debajo (optimista) y la CV espacial queda por encima (conservadora):

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(tabla))
ax.bar(x, tabla['RMSE_medio'], yerr=tabla['RMSE_sd'], capsize=8,
       color=['#2E86AB','#A23B72'], alpha=0.85)
ax.axhline(0.6073, color='red', ls='--', lw=2, label='RMSE test grid (referencia)')
ax.set_xticks(x); ax.set_xticklabels(tabla['esquema'])
ax.set_ylabel('RMSE k-fold')
ax.set_title('CV aleatoria vs CV espacial — gap respecto al test')
ax.legend()
for i, (m, s, gap) in enumerate(zip(tabla['RMSE_medio'], tabla['RMSE_sd'], tabla['gap_vs_test_pct'])):
    ax.text(i, m+s+0.03, f'gap={gap:+.1f}%', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig7_cv_espacial_vs_aleatoria.png'), dpi=120, bbox_inches='tight')
plt.show()

### Lectura

La CV aleatoria entrega RMSE $\approx 0{,}58$ ($-4\%$ respecto al test), es decir, le miente al usuario sugiriendo mejor desempeño del real. La CV espacial entrega $\approx 0{,}70$ ($+15{,}5\%$ respecto al test), conservadora pero honesta porque evalúa explícitamente la capacidad de extrapolar a zonas espacialmente independientes. Para reportar desempeño en un Digital Twin de monitoreo ambiental — donde la pregunta es predecir en zonas no muestreadas — la CV espacial es el estimador correcto.